In [ ]:
import os
import random
from pathlib import Path
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

# ----------------- تنظیم تکرارپذیری -----------------
def seed_everything(seed: int = 2025):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(2025)

# ----------------- تنظیم دستگاه -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ----------------- ماژول‌های خودت -----------------
import importlib
import utilityFunctions
importlib.reload(utilityFunctions)
from utilityFunctions import reshapeData, prepare_data_sliding_window,roi_gate_regularizer,schedule_reg,proto_losses

import FXPNet_Models
importlib.reload(FXPNet_Models)
from FXPNet_Models import ConvNet21, FuzzyProtoNet21V3Light

# همه‌ی مدل‌هایی که می‌خواهی تست کنی
MODEL_DICT = {
    "FuzzyProtoNet21V3Light": FuzzyProtoNet21V3Light
    "ConvNet21": ConvNet21,
}

# سه اطلس
ATLASES = ["Brainnetome", "Gordon", "Glasser"]

# هایپرپارامترها
WINDOW_SIZE = 256
STEP        = 128
N_SPLITS    = 5
BATCH_SIZE  = 32
LR          = 1e-4
NUM_EPOCHS  = 20

# ----------------- مسیرهای اصلی -----------------
BASE_DIR   = Path(r"F:/GenderSimplex_Project/HCP_1200")
LABEL_CSV  = BASE_DIR / "subjects_sex.csv"
SUBJ_TXT   = BASE_DIR / "Final_List_Subjects_HCP.txt"  # سوژه‌هایی که 4 ران خام دارند

# =================================================
# 1) خواندن لیبل‌ها (مشترک بین همه اطلس‌ها)
# =================================================
df_lab = pd.read_csv(LABEL_CSV)
df_lab["Subject"] = df_lab["Subject"].astype(str)
gender_map = {"M": 0, "F": 1, "Male": 0, "Female": 1}
df_lab["Label"] = df_lab["Sex"].map(gender_map)
label_dict = dict(zip(df_lab["Subject"], df_lab["Label"]))

# =================================================
# 2) خواندن لیست سوژه‌ها از فایل متنی
# =================================================
with open(SUBJ_TXT, "r", encoding="utf-8") as f:
    subjects_all = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(subjects_all)} subject IDs from text list.")

RUNS = [1, 2, 3, 4]


# =================================================
# حلقه‌ی اصلی روی اطلس‌ها
# =================================================
for ATLAS_NAME in ATLASES:
    print("\n==============================")
    print(f"### Atlas = {ATLAS_NAME}")
    print("==============================")

    # مسیر سیگنال‌های پارسل‌شده‌ی این اطلس
    BASE_DIR_STANDARD = BASE_DIR / "Parcellated" / ATLAS_NAME

    valid_sids = []
    labels_list = []
    ts_dict = {}   # ts_dict[sid][run_id] = (T, N_roi)

    # ---------------------------------------------
    # 2-1) جمع کردن سیگنال‌های زمانی برای این اطلس
    # ---------------------------------------------
    for sid in subjects_all:
        if sid not in label_dict:
            continue

        all_runs_ok = True
        runs_ts = {}

        for run_id in RUNS:
            # شما "Standard" ندارید ولی من همان شرط شما را نگه داشتم
            if ATLAS_NAME == "Standard":
                npy_path = BASE_DIR_STANDARD / f"temp_ts_{sid}" / f"{sid}_run{run_id}_ts.npy"
            else:
                npy_path = BASE_DIR_STANDARD / f"{sid}_run{run_id}_ts.npy"

            if not npy_path.exists():
                all_runs_ok = False
                break

            X = np.load(npy_path)   # (T, N_roi)
            runs_ts[run_id] = X

        if not all_runs_ok:
            continue

        valid_sids.append(sid)
        labels_list.append(label_dict[sid])
        ts_dict[sid] = runs_ts

    labels = np.array(labels_list, dtype=int)
    print(f"Atlas {ATLAS_NAME}: usable subjects with 4 runs & labels = {len(valid_sids)}")

    if len(valid_sids) == 0:
        continue

    # ---------------------------------------------
    # 2-2) k-fold روی سوژه‌ها (برای این اطلس)
    # ---------------------------------------------
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=2025)
    fold_splits = list(skf.split(np.arange(len(valid_sids)), labels))

    # ---------------------------------------------
    # 2-3) ساخت data_all برای هر run
    # ---------------------------------------------
    def build_data_for_run(run_id, valid_sids, ts_dict):
        ts_list = []
        for sid in valid_sids:
            X = ts_dict[sid][run_id]   # (T, N_roi)

            # Z-score روی زمان برای هر ROI
            mean_t = X.mean(axis=0, keepdims=True)
            std_t  = X.std(axis=0,  keepdims=True)
            std_t[std_t == 0] = 1.0
            X_norm = (X - mean_t) / std_t
            ts_list.append(X_norm)

        # هم‌طول کردن زمان
        T_min = min(ts.shape[0] for ts in ts_list)
        ts_list = [ts[:T_min, :] for ts in ts_list]
        data_all = np.stack(ts_list, axis=0).astype(np.float32)  # (N_subj, T_min, N_roi)

        # Z-score ثانویه به ازای هر سوژه
        mean = data_all.mean(axis=1, keepdims=True)
        std  = data_all.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        data_all = (data_all - mean) / std

        return data_all, T_min

    # ---------------------------------------------
    # 2-4) cache داده برای هر run
    # ---------------------------------------------
    run_data_cache = {}
    for run_id in RUNS:
        data_all, T_min = build_data_for_run(run_id, valid_sids, ts_dict)
        run_data_cache[run_id] = (data_all, T_min)
        print(f"Atlas {ATLAS_NAME} → prepared data for run {run_id}: shape={data_all.shape}, T_min={T_min}")

    # =================================================
    # 3) حلقه‌ی مدل‌ها برای این اطلس
    # =================================================
    for MODEL_NAME, MODEL_CLS in MODEL_DICT.items():
        print("\n--------------------------------")
        print(f"### Atlas = {ATLAS_NAME} | Model = {MODEL_NAME}")
        print("--------------------------------")

        OUT_MODELS_DIR = BASE_DIR / "Final_AllResults_V2" / "saved_models" / ATLAS_NAME / MODEL_NAME
        OUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

        results_rows = []

        # ---------------------------------------------
        # 3-1) حلقه‌ی run ها
        # ---------------------------------------------
        for run_id in RUNS:
            print(f"\n==============================")
            print(f"Scenario: ATLAS={ATLAS_NAME}, MODEL={MODEL_NAME}, RUN={run_id}")
            print(f"==============================")

            data_all_run, T_run = run_data_cache[run_id]

            if WINDOW_SIZE > T_run:
                raise ValueError(f"WINDOW_SIZE={WINDOW_SIZE} > T_run={T_run}")

            data_all_run = data_all_run[:, :T_run, :]  # (N_subj, T, N_roi)
            N_roi = data_all_run.shape[2]

            # ---- حلقه‌ی فولدها ----
            for fold_id, (train_idx, val_idx) in enumerate(fold_splits, start=1):
                print(f"\n---------- Fold {fold_id}/{N_SPLITS} (Train/Val) ----------")

                y_train_subj = labels[train_idx]
                y_val_subj   = labels[val_idx]

                data_train = data_all_run[train_idx]   # (N_train, T, N_roi)
                data_val   = data_all_run[val_idx]     # (N_val,   T, N_roi)

                # sliding window + subj index
                X_train_win, y_train_win, train_subj_idx_win = prepare_data_sliding_window(
                    data_train, y_train_subj,
                    window_size=WINDOW_SIZE, step=STEP,
                    return_subj_index=True
                )
                X_val_win, y_val_win, val_subj_idx_win = prepare_data_sliding_window(
                    data_val, y_val_subj,
                    window_size=WINDOW_SIZE, step=STEP,
                    return_subj_index=True
                )

                print(f"Train windows: {X_train_win.shape[0]}, Val windows: {X_val_win.shape[0]}")
                if X_train_win.shape[0] == 0 or X_val_win.shape[0] == 0:
                    print("⚠ No windows produced (check WINDOW_SIZE/STEP vs T). Skipping fold.")
                    continue

                # reshape به (samples, C, T) برای Conv1d
                X_train_reshape = reshapeData(X_train_win)  # (N_win, N_roi, W)
                X_val_reshape   = reshapeData(X_val_win)

                # DataLoader
                train_dataset = TensorDataset(
                    torch.from_numpy(X_train_reshape).float(),
                    torch.from_numpy(y_train_win).long()
                )
                val_dataset = TensorDataset(
                    torch.from_numpy(X_val_reshape).float(),
                    torch.from_numpy(y_val_win).long()
                )

                train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
                val_loader   = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False)

                # -------------------------------------------------
                # مدل، Loss, Optimizer
                # -------------------------------------------------
                n_roi = X_train_reshape.shape[1]
                model = MODEL_CLS(n_roi=n_roi).to(device)

                optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)

                # ✅ class weights فقط از train fold (ترجیحاً بر اساس پنجره‌های train)
                cw = compute_class_weight(
                    class_weight="balanced",
                    classes=np.array([0, 1]),
                    y=y_train_win
                )
                class_weights = torch.tensor(cw, dtype=torch.float32, device=device)
                criterion = nn.CrossEntropyLoss(weight=class_weights)

                best_val_subj_acc = -1.0
                best_state = None

                # ----------------- Training loop -----------------
                best_eval_acc = 0.0
                best_state    = None

                for epoch in range(1, NUM_EPOCHS + 1):
                    # ------------------- Train -------------------
                    model.train()
                    train_losses = []
                    train_ce_losses = []
                    train_reg_losses = []

                    l1_alpha, ent_beta = schedule_reg(epoch, warmup_epochs=5)

                    for xb, yb in train_loader:
                        xb = xb.to(device)
                        yb = yb.to(device)

                        optimizer.zero_grad()

                        # نکته: در train return_details=True تا roi_importance detach نشده داشته باشیم
                        logits, details = model(xb, return_details=True)
                        ce_loss = criterion(logits, yb)

                        p_loss, p_terms = proto_losses(details, yb,
                                                    w_clust=1e-2, w_sep=1e-2, w_div=1e-3, w_sparse=1e-3)

                        loss = ce_loss + p_loss
                        loss.backward()
                        optimizer.step()

                        train_losses.append(loss.item())
                        train_ce_losses.append(ce_loss.item())


                    # ------------------- Eval (VAL) -------------------
                    model.eval()
                    all_logits_eval = []
                    all_y_eval      = []
                    with torch.no_grad():
                        for xb, yb in val_loader:
                            xb = xb.to(device)
                            yb = yb.to(device)

                            # برای eval سریع‌تر: return_details=False
                            logits = model(xb, return_details=False)

                            all_logits_eval.append(logits.detach().cpu())
                            all_y_eval.append(yb.detach().cpu())

                    if len(all_logits_eval) == 0:
                        eval_acc = 0.0
                    else:
                        all_logits_eval = torch.cat(all_logits_eval, dim=0)
                        all_y_eval      = torch.cat(all_y_eval, dim=0)

                        y_pred_eval = all_logits_eval.argmax(dim=1).numpy()
                        y_true_eval = all_y_eval.numpy()

                        eval_acc = accuracy_score(y_true_eval, y_pred_eval)

                    print(
                        f"Epoch {epoch:03d} | "
                        f"TrainLoss={float(np.mean(train_losses)):.4f} "
                        f"CE={float(np.mean(train_ce_losses)):.4f}, "
                        f"EvalAcc={eval_acc:.3f}"
                    )

                    if eval_acc > best_eval_acc:
                        best_eval_acc = eval_acc
                        best_state    = copy.deepcopy(model.state_dict())

                # fallback
                if best_state is None:
                    best_state = copy.deepcopy(model.state_dict())

                model.load_state_dict(best_state)


                # ذخیره بهترین مدل این فولد
                fold_model_path = OUT_MODELS_DIR / f"{MODEL_NAME}_run{run_id}_fold{fold_id}.pt"
                torch.save(best_state, fold_model_path)
                print(f"Saved best model for fold {fold_id} → {fold_model_path}")

                # ----------------- Final Val metrics با بهترین مدل -----------------
                model.eval()
                all_logits_val = []
                all_y_val      = []
                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb = xb.to(device)
                        yb = yb.to(device)
                        logits = model(xb)
                        all_logits_val.append(logits.cpu())
                        all_y_val.append(yb.cpu())

                all_logits_val = torch.cat(all_logits_val, dim=0)
                all_y_val      = torch.cat(all_y_val, dim=0)

                y_pred_win = all_logits_val.argmax(dim=1).numpy()
                y_true_win = all_y_val.numpy()

                # window-level metrics
                acc_win  = accuracy_score(y_true_win, y_pred_win)
                f1_win   = f1_score(y_true_win, y_pred_win)
                prec_win = precision_score(y_true_win, y_pred_win)
                rec_win  = recall_score(y_true_win, y_pred_win)
                bacc_win = balanced_accuracy_score(y_true_win, y_pred_win)

                print(
                    f"[RUN={run_id}, fold={fold_id}] Val (window-level) → "
                    f"Acc={acc_win:.3f}, F1={f1_win:.3f}, Prec={prec_win:.3f}, "
                    f"Rec={rec_win:.3f}, BAcc={bacc_win:.3f}"
                )

                # subject-level metrics (robust)
                subj_pred = []
                subj_true = []
                subj_ids_val = np.array(valid_sids)[val_idx]

                for local_s, sid in enumerate(subj_ids_val):
                    mask = (val_subj_idx_win == local_s)
                    preds_s = y_pred_win[mask]
                    vote = int(preds_s.mean() >= 0.5)
                    subj_pred.append(vote)
                    subj_true.append(y_val_subj[local_s])

                subj_pred = np.array(subj_pred)
                subj_true = np.array(subj_true)

                acc_subj  = accuracy_score(subj_true, subj_pred)
                f1_subj   = f1_score(subj_true, subj_pred)
                prec_subj = precision_score(subj_true, subj_pred)
                rec_subj  = recall_score(subj_true, subj_pred)
                bacc_subj = balanced_accuracy_score(subj_true, subj_pred)

                print(
                    f"[RUN={run_id}, fold={fold_id}] Val (subject-level) → "
                    f"Acc={acc_subj:.3f}, F1={f1_subj:.3f}, Prec={prec_subj:.3f}, "
                    f"Rec={rec_subj:.3f}, BAcc={bacc_subj:.3f}"
                )

                results_rows.append({
                    "atlas": ATLAS_NAME,
                    "model": MODEL_NAME,
                    "run":   run_id,
                    "fold":  fold_id,
                    "acc_win":   acc_win,
                    "f1_win":    f1_win,
                    "prec_win":  prec_win,
                    "rec_win":   rec_win,
                    "bacc_win":  bacc_win,
                    "acc_subj":  acc_subj,
                    "f1_subj":   f1_subj,
                    "prec_subj": prec_subj,
                    "rec_subj":  rec_subj,
                    "bacc_subj": bacc_subj,
                    "best_val_subj_acc_during_training": best_val_subj_acc,
                })

        # خروجی CSV برای این (مدل, اطلس)
        results_df = pd.DataFrame(results_rows)
        out_csv = BASE_DIR / "Final_AllResults_V2" / f"{MODEL_NAME}_{ATLAS_NAME}_runwise_5fold_val.csv"
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        results_df.to_csv(out_csv, index=False)
        print("\nSaved all run/fold val results to:", out_csv)


In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

# =========================
# 1) Set your folder
# =========================
CSV_DIR = Path(r"F:/GenderSimplex_Project/HCP_1200/Final_AllResults_V2")  # <-- این را تغییر بده
OUT_CSV = CSV_DIR / "summary_algo_atlas_run_subject_mean_var.csv"

# =========================
# 2) Helpers
# =========================
def parse_algo_atlas_from_filename(name: str):
    """
    Expected examples:
      ConvNet21_Brainnetome_runwise_5fold_val.csv
      FuzzyProtoNet21V3Light_Gordon_runwise_5fold_val.csv
    """
    m = re.match(r"^(?P<algo>.+)_(?P<atlas>Brainnetome|Gordon|Glasser)_runwise_5fold_val\.csv$", name)
    if not m:
        return None, None
    return m.group("algo"), m.group("atlas")

def detect_run_col(df: pd.DataFrame):
    for c in ["run", "Run", "RUN", "run_id", "RUN_ID"]:
        if c in df.columns:
            return c
    return None

def choose_subject_metric_cols(df: pd.DataFrame):
    """
    سعی می‌کند ستون(های) مربوط به نتایج سطح subject را پیدا کند.
    اگر ستون‌های واضح نبود، همه ستون‌های عددی (به جز run/fold/epoch/seed) را برمی‌دارد.
    """
    # candidates by name
    patterns = [
        r"subject", r"subj",
        r"acc", r"accuracy",
        r"f1", r"auc", r"bal", r"balanced",
        r"precision", r"recall",
        r"sens", r"spec"
    ]
    lower_cols = {c: c.lower() for c in df.columns}

    # numeric columns
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    # remove non-metric columns
    drop_like = {"fold", "Fold", "FOLD", "epoch", "Epoch", "seed", "Seed"}
    numeric_cols = [c for c in numeric_cols if c not in drop_like]

    # prioritize columns whose names suggest subject-level metric
    scored = []
    for c in numeric_cols:
        s = lower_cols[c]
        score = 0
        if "subject" in s or "subj" in s:
            score += 5
        if "acc" in s or "accuracy" in s:
            score += 3
        if "f1" in s:
            score += 2
        if "auc" in s:
            score += 2
        # general signal
        for p in patterns:
            if re.search(p, s):
                score += 1
        scored.append((score, c))

    scored.sort(reverse=True)
    # keep top metric columns with score>=3; if none, fallback to all numeric (except run)
    metric_cols = [c for sc, c in scored if sc >= 3]

    return metric_cols if len(metric_cols) > 0 else numeric_cols

# =========================
# 3) Read and summarize
# =========================
rows = []

csv_files = sorted(CSV_DIR.glob("*_runwise_5fold_val.csv"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No '*_runwise_5fold_val.csv' found in: {CSV_DIR}")

for fp in csv_files:
    algo, atlas = parse_algo_atlas_from_filename(fp.name)
    if algo is None:
        # اگر اسم فایل با الگو جور نیست، ردش می‌کنیم
        continue

    df = pd.read_csv(fp)

    run_col = detect_run_col(df)
    if run_col is None:
        raise ValueError(f"[{fp.name}] No run column found. Please add a 'run' column or rename it.")

    metric_cols = choose_subject_metric_cols(df)
    # اگر ستون run در metric_cols افتاده بود حذفش کن
    metric_cols = [c for c in metric_cols if c != run_col]

    if len(metric_cols) == 0:
        raise ValueError(f"[{fp.name}] No numeric metric columns detected.")

    # group by run and compute mean/var
    for run_id, g in df.groupby(run_col):
        out = {
            "algorithm": algo,
            "atlas": atlas,
            "run": int(run_id),
            "n_rows": int(len(g)),
        }
        for mc in metric_cols:
            vals = pd.to_numeric(g[mc], errors="coerce").dropna().values
            if len(vals) == 0:
                out[f"{mc}_mean"] = np.nan
                out[f"{mc}_std"]  = np.nan
            else:
                mean = np.mean(vals)
                std  = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
                out[f"{mc}_mean"] = round(100*mean, 3)
                out[f"{mc}_std"]  = round(100*std, 3)
                out[f"{mc}_mean_pm_std"] = f"{100*mean:.3f} ± {100*std:.3f}"
        rows.append(out)

summary = pd.DataFrame(rows)

# مرتب‌سازی تمیز
summary = summary.sort_values(["algorithm", "atlas", "run"]).reset_index(drop=True)

# =========================
# 4) Save and (optional) view
# =========================
summary.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
print(summary.head(20))

# اگر یک جدول Pivot هم خواستی (فقط برای متریک اصلی مثل subject_acc):
# اینجا اگر ستون subject_acc_mean وجود داشت، pivot می‌کنیم:
main_candidates = [c for c in summary.columns if c.endswith("_mean") and ("acc" in c.lower() and ("subj" in c.lower() or "subject" in c.lower()))]
if len(main_candidates) > 0:
    main = main_candidates[0]
    piv = summary.pivot_table(index=["algorithm","atlas"], columns="run", values=main, aggfunc="mean")
    piv = piv.reset_index()
    piv_path = CSV_DIR / f"pivot_{main}.csv"
    piv.to_csv(piv_path, index=False)
    print("Also saved pivot:", piv_path)


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

# ----------------------------
# Config
# ----------------------------
BASE = Path(r"F:\GenderSimplex_Project\HCP_1200\Final_AllResults_V2")

FILES = {
    "Brainnetome": {
        "ConvNet": BASE / "ConvNet21_Brainnetome_runwise_5fold_val.csv",
        "FXPNet":  BASE / "FuzzyProtoNet21V3Light_Brainnetome_runwise_5fold_val.csv",
    },
    "Glasser": {
        "ConvNet": BASE / "ConvNet21_Glasser_runwise_5fold_val.csv",
        "FXPNet":  BASE / "FuzzyProtoNet21V3Light_Glasser_runwise_5fold_val.csv",
    },
    "Gordon": {
        "ConvNet": BASE / "ConvNet21_Gordon_runwise_5fold_val.csv",
        "FXPNet":  BASE / "FuzzyProtoNet21V3Light_Gordon_runwise_5fold_val.csv",
    },
}

# subject-level metrics in your CSV
SUBJ_METRICS = ["acc_subj", "f1_subj", "prec_subj", "rec_subj", "bacc_subj"]

ALPHA = 0.05

# ----------------------------
# Helpers
# ----------------------------
def fdr_bh(pvals):
    """Benjamini–Hochberg FDR. Returns q-values in original order."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * n / (np.arange(n) + 1.0)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out = np.empty_like(q)
    out[order] = q
    return out

def cohen_dz(diff):
    """Effect size for paired samples: dz = mean(diff)/std(diff)."""
    diff = np.asarray(diff, dtype=float)
    sd = np.std(diff, ddof=1)
    if sd < 1e-12:
        return np.nan
    return float(np.mean(diff) / sd)

def paired_tests(x, y):
    """
    x,y: paired arrays
    returns: t-test p, wilcoxon p, mean diff, dz
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    d = x - y

    # paired t-test
    t_stat, p_t = stats.ttest_rel(x, y, nan_policy="omit")

    # Wilcoxon signed-rank (robust)
    try:
        w_stat, p_w = stats.wilcoxon(d)
    except Exception:
        p_w = np.nan

    return float(p_t), float(p_w), float(np.mean(d)), cohen_dz(d)

def load_one(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing file: {csv_path}")
    df = pd.read_csv(csv_path)
    # basic sanity
    needed = {"atlas", "model", "run", "fold"} | set(SUBJ_METRICS)
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{csv_path.name} missing columns: {missing}")
    return df

# ----------------------------
# Main
# ----------------------------
rows_runwise = []   # per Atlas × Run (n=5 folds)
rows_atlas = []     # per Atlas pooled (n=20)

for atlas, paths in FILES.items():
    df_conv = load_one(paths["ConvNet"]).copy()
    df_fxp  = load_one(paths["FXPNet"]).copy()

    # keep only this atlas (safety)
    df_conv = df_conv[df_conv["atlas"].astype(str) == atlas]
    df_fxp  = df_fxp[df_fxp["atlas"].astype(str) == atlas]

    # merge on run & fold to create paired data
    key = ["run", "fold"]
    merged = pd.merge(
        df_fxp[key + SUBJ_METRICS],
        df_conv[key + SUBJ_METRICS],
        on=key,
        suffixes=("_fxp", "_conv"),
        how="inner"
    )

    if merged.empty:
        raise RuntimeError(f"No paired rows found for atlas={atlas}. Check run/fold columns.")

    # ---- (1) Run-wise tests (per run, n=5 folds)
    for run_id in sorted(merged["run"].unique()):
        sub = merged[merged["run"] == run_id].sort_values("fold")
        if len(sub) < 3:
            continue

        for m in SUBJ_METRICS:
            x = sub[f"{m}_fxp"].to_numpy()
            y = sub[f"{m}_conv"].to_numpy()

            p_t, p_w, mean_diff, dz = paired_tests(x, y)
            rows_runwise.append({
                "atlas": atlas,
                "run": int(run_id),
                "n_pairs": int(len(sub)),
                "metric": m,
                "mean_fxp": float(np.mean(x)),
                "mean_conv": float(np.mean(y)),
                "mean_diff_fxp_minus_conv": mean_diff,
                "cohens_dz": dz,
                "p_paired_t": p_t,
                "p_wilcoxon": p_w,
            })

    # ---- (2) Atlas-wise pooled tests (all runs×folds, n=20)
    pooled = merged.sort_values(["run", "fold"])
    for m in SUBJ_METRICS:
        x = pooled[f"{m}_fxp"].to_numpy()
        y = pooled[f"{m}_conv"].to_numpy()

        p_t, p_w, mean_diff, dz = paired_tests(x, y)
        rows_atlas.append({
            "atlas": atlas,
            "scope": "pooled_runs_folds",
            "n_pairs": int(len(pooled)),
            "metric": m,
            "mean_fxp": float(np.mean(x)),
            "mean_conv": float(np.mean(y)),
            "mean_diff_fxp_minus_conv": mean_diff,
            "cohens_dz": dz,
            "p_paired_t": p_t,
            "p_wilcoxon": p_w,
        })

df_runwise = pd.DataFrame(rows_runwise)
df_atlas   = pd.DataFrame(rows_atlas)

# ----------------------------
# Multiple-comparison correction (FDR) separately for runwise and pooled
# ----------------------------
if not df_runwise.empty:
    df_runwise["q_fdr_bh_t"] = fdr_bh(df_runwise["p_paired_t"].to_numpy())
    df_runwise["sig_fdr_t"] = df_runwise["q_fdr_bh_t"] < ALPHA

if not df_atlas.empty:
    df_atlas["q_fdr_bh_t"] = fdr_bh(df_atlas["p_paired_t"].to_numpy())
    df_atlas["sig_fdr_t"] = df_atlas["q_fdr_bh_t"] < ALPHA

# nicer sorting
if not df_runwise.empty:
    df_runwise = df_runwise.sort_values(["metric", "atlas", "run"]).reset_index(drop=True)
df_atlas = df_atlas.sort_values(["metric", "atlas"]).reset_index(drop=True)

# save
out_runwise = BASE / "stats_subjectlevel_runwise_paired_tests.csv"
out_atlas   = BASE / "stats_subjectlevel_atlas_pooled_paired_tests.csv"
df_runwise.to_csv(out_runwise, index=False)
df_atlas.to_csv(out_atlas, index=False)

print("✅ Saved:", out_runwise)
print("✅ Saved:", out_atlas)

# show quick summary for bacc_subj
print("\n=== POOLED (atlas-wise) summary for bacc_subj ===")
print(df_atlas[df_atlas["metric"]=="bacc_subj"][["atlas","mean_conv","mean_fxp","mean_diff_fxp_minus_conv","p_paired_t","q_fdr_bh_t","sig_fdr_t","cohens_dz"]])
